# V3: prior-comorbidity ICD embedding

Three ablation groups: static_lr → burden_lr → embedding_lr.

burden includes the number of prior hospitalizations, the number of distinct exact codes, the number of diagnosis events and whether there is any prior diagnosis; the first three are compressed with log1p. embedding_lr adds TF-IDF + TruncatedSVD (at most 32 dimensions) on top of burden_lr, to separate the gain from "richer diagnoses" from that of the "latent representation".

The vocabulary, IDF, SVD and scaling are fitted only on the training subset; the validation set is only transformed. No history, an empty vocabulary and a single-term vocabulary all have fallback logic that keeps the patient. Unknown codes in validation do not extend the vocabulary.

This is a structured diagnosis representation, not clinical notes; there are currently no clinical text data. The static CLIN comorbidities still inherit the V3 assumption of prior ∪ index chronic codes, while the new embedding uses only prior hospitalizations that have ended. Deployment would still need to state whether the original static codes are available at 24 hours.

**balanced simplified-compute version**: fixed base-model parameters, full keeps the 5×10 outer validation; LASSO inner 5 folds/30 λ, stacking inner 3 folds. Uses a cache separate from the parent strict version; the old predictions of the first 6 folds are not mixed in. No real patient model of the new version has been run yet.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the project from the extension directory or the notebooks subdirectory, to avoid hard-coding a Windows drive letter.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dn_extensions" / "settings.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("start from dn_mortality_repro_V3_extensions or its notebooks subdirectory.")
sys.path.insert(0, str(ROOT))
print("extension directory:", ROOT)


In [ ]:
# pilot is a small 1×3 outer-fold trial run; for the formal experiment set PROFILE to "full".
# Starts only when RUN_EXPERIMENT is set to True; full may take a long time.
PROFILE = "pilot"
RUN_EXPERIMENT = False
EXPERIMENTS = ['static_lr', 'burden_lr', 'embedding_lr']
if RUN_EXPERIMENT:
    from dn_extensions.evaluation import run_evaluation
    from dn_extensions.settings import profile_options
    output, summary, comparisons = run_evaluation(profile_options(PROFILE), EXPERIMENTS)
    display(summary)
    display(comparisons)
else:
    print("model training in this notebook has not been started.")


In [ ]:
# Show the formal full results first; without results print the status, no retraining.
RESULT_PROFILE = "full"
latest = ROOT / "results" / RESULT_PROFILE / "latest.txt"
if latest.exists():
    folder = latest.parent / latest.read_text().strip()
    summary = pd.read_csv(folder / "summary.csv")
    display(summary[summary.experiment.isin(EXPERIMENTS)])
    print("detailed record:", folder / "report.md")
else:
    print("no experiment results for this mode yet. Set RESULT_PROFILE to pilot to show pilot results.")


## Experiment status and conclusions (updated automatically)

Run mode: `full`; method: balanced fixed parameters; result directory: `results/full/27464ba98305edbc`.

LASSO 5 folds/30 λ, stacking 3 folds; static and extension models use the same simplified method. No folds of the strict version were mixed in.

The fixed held-out set took no part in this experiment. Tree-model extensions need no internal stacking cross-fitting; the original fixed parameters and within-training-fold preprocessing are unchanged.

The Holm correction covers the paired comparisons currently available; corrected p-values may change when experiments are added later.

```text

   experiment  n_outer_folds  AUROC_fold_mean  AUROC_fold_sd  AUPRC_fold_mean  Brier_fold_mean  calibration_slope_fold_mean  seconds_total  AUROC_pooled_mean  AUPRC_pooled_mean  Brier_pooled_mean  cal_intercept_pooled_mean  cal_slope_pooled_mean
    burden_lr             50          0.79936        0.02666          0.60660          0.15087                      0.91420      578.69442            0.79925            0.59466            0.15086                    0.00359                0.89227
    burden_rf             50          0.79724        0.02871          0.60343          0.15232                      1.22278       28.27988            0.79581            0.58966            0.15232                    0.00337                1.18404
   burden_xgb             50          0.79396        0.03022          0.60583          0.15265                      0.93979        6.26590            0.79309            0.59511            0.15264                    0.00653                0.91576
 embedding_lr             50          0.79591        0.02741          0.60562          0.15229                      0.85161      582.82427            0.79555            0.59278            0.15228                    0.00813                0.83040
 embedding_rf             50          0.79965        0.02798          0.60840          0.15269                      1.39541       32.57728            0.79825            0.59542            0.15268                    0.00490                1.35224
embedding_xgb             50          0.79429        0.02760          0.60649          0.15232                      0.97897        9.29661            0.79331            0.59430            0.15231                   -0.00756                0.95188
    static_lr             50          0.79928        0.02648          0.60740          0.15092                      0.92073      502.87958            0.79885            0.59482            0.15091                    0.00385                0.89831
    static_rf             50          0.79798        0.02809          0.60618          0.15177                      1.19841       15.22184            0.79671            0.59435            0.15177                    0.00179                1.16211
   static_xgb             50          0.79504        0.02965          0.60572          0.15250                      0.94083        4.61352            0.79408            0.59662            0.15249                    0.00766                0.91690

```

Paired differences:

```text

   experiment  reference metric  mean_diff  sd_diff  se_corrected       lo      hi        t  df  p_value  k  r  p_holm_current_family
    static_rf  static_lr  AUROC   -0.00130  0.01457       0.00528 -0.01190 0.00931 -0.24552  49  0.80708 10  5                    1.0
    static_rf  static_lr  AUPRC   -0.00122  0.02900       0.01050 -0.02232 0.01989 -0.11569  49  0.90837 10  5                    1.0
    static_rf  static_lr  Brier    0.00085  0.00469       0.00170 -0.00256 0.00427  0.50207  49  0.61787 10  5                    1.0
   static_xgb  static_lr  AUROC   -0.00423  0.01409       0.00510 -0.01449 0.00602 -0.82968  49  0.41074 10  5                    1.0
   static_xgb  static_lr  AUPRC   -0.00167  0.02894       0.01048 -0.02273 0.01938 -0.15972  49  0.87376 10  5                    1.0
   static_xgb  static_lr  Brier    0.00158  0.00519       0.00188 -0.00220 0.00536  0.83926  49  0.40540 10  5                    1.0
    burden_lr  static_lr  AUROC    0.00008  0.00315       0.00114 -0.00221 0.00237  0.07071  49  0.94392 10  5                    1.0
    burden_lr  static_lr  AUPRC   -0.00080  0.00669       0.00242 -0.00566 0.00407 -0.32854  49  0.74391 10  5                    1.0
    burden_lr  static_lr  Brier   -0.00005  0.00109       0.00039 -0.00084 0.00074 -0.12687  49  0.89956 10  5                    1.0
 embedding_lr  burden_lr  AUROC   -0.00345  0.01003       0.00363 -0.01075 0.00385 -0.94986  49  0.34685 10  5                    1.0
 embedding_lr  burden_lr  AUPRC   -0.00098  0.01402       0.00508 -0.01118 0.00922 -0.19368  49  0.84723 10  5                    1.0
 embedding_lr  burden_lr  Brier    0.00142  0.00345       0.00125 -0.00109 0.00393  1.13866  49  0.26038 10  5                    1.0
    burden_rf  static_rf  AUROC   -0.00074  0.00616       0.00223 -0.00522 0.00374 -0.33253  49  0.74091 10  5                    1.0
    burden_rf  static_rf  AUPRC   -0.00275  0.01287       0.00466 -0.01212 0.00661 -0.59075  49  0.55740 10  5                    1.0
    burden_rf  static_rf  Brier    0.00055  0.00173       0.00063 -0.00071 0.00181  0.87819  49  0.38412 10  5                    1.0
   burden_xgb static_xgb  AUROC   -0.00108  0.00676       0.00245 -0.00600 0.00384 -0.44121  49  0.66100 10  5                    1.0
   burden_xgb static_xgb  AUPRC    0.00010  0.01749       0.00633 -0.01263 0.01283  0.01602  49  0.98728 10  5                    1.0
   burden_xgb static_xgb  Brier    0.00015  0.00214       0.00077 -0.00140 0.00171  0.19740  49  0.84433 10  5                    1.0
 embedding_rf  burden_rf  AUROC    0.00241  0.00885       0.00320 -0.00403 0.00885  0.75282  49  0.45516 10  5                    1.0
 embedding_rf  burden_rf  AUPRC    0.00497  0.01607       0.00582 -0.00673 0.01666  0.85397  49  0.39728 10  5                    1.0
 embedding_rf  burden_rf  Brier    0.00037  0.00239       0.00087 -0.00137 0.00211  0.42331  49  0.67392 10  5                    1.0
embedding_xgb burden_xgb  AUROC    0.00033  0.01064       0.00385 -0.00741 0.00807  0.08550  49  0.93221 10  5                    1.0
embedding_xgb burden_xgb  AUPRC    0.00067  0.02335       0.00846 -0.01632 0.01766  0.07910  49  0.93728 10  5                    1.0
embedding_xgb burden_xgb  Brier   -0.00033  0.00395       0.00143 -0.00321 0.00254 -0.23263  49  0.81702 10  5                    1.0
 embedding_rf  static_rf  AUROC    0.00167  0.00963       0.00349 -0.00534 0.00868  0.47943  49  0.63377 10  5                    1.0
 embedding_rf  static_rf  AUPRC    0.00222  0.01939       0.00702 -0.01189 0.01632  0.31582  49  0.75348 10  5                    1.0
 embedding_rf  static_rf  Brier    0.00092  0.00260       0.00094 -0.00098 0.00281  0.97278  49  0.33544 10  5                    1.0
embedding_xgb static_xgb  AUROC   -0.00075  0.00988       0.00358 -0.00794 0.00644 -0.20974  49  0.83474 10  5                    1.0
embedding_xgb static_xgb  AUPRC    0.00077  0.02388       0.00865 -0.01661 0.01815  0.08909  49  0.92938 10  5                    1.0
embedding_xgb static_xgb  Brier   -0.00018  0.00389       0.00141 -0.00301 0.00265 -0.12793  49  0.89873 10  5                    1.0

```

- `static_rf` vs `static_lr`: ΔAUROC=-0.0013, corrected 95% CI [-0.0119, 0.0093]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `static_xgb` vs `static_lr`: ΔAUROC=-0.0042, corrected 95% CI [-0.0145, 0.0060]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `burden_lr` vs `static_lr`: ΔAUROC=0.0001, corrected 95% CI [-0.0022, 0.0024]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `embedding_lr` vs `burden_lr`: ΔAUROC=-0.0035, corrected 95% CI [-0.0108, 0.0038]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `burden_rf` vs `static_rf`: ΔAUROC=-0.0007, corrected 95% CI [-0.0052, 0.0037]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `burden_xgb` vs `static_xgb`: ΔAUROC=-0.0011, corrected 95% CI [-0.0060, 0.0038]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `embedding_rf` vs `burden_rf`: ΔAUROC=0.0024, corrected 95% CI [-0.0040, 0.0089]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `embedding_xgb` vs `burden_xgb`: ΔAUROC=0.0003, corrected 95% CI [-0.0074, 0.0081]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `embedding_rf` vs `static_rf`: ΔAUROC=0.0017, corrected 95% CI [-0.0053, 0.0087]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `embedding_xgb` vs `static_xgb`: ΔAUROC=-0.0008, corrected 95% CI [-0.0079, 0.0064]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

## 2026-10-14 review of the comorbidity ablation

static_lr, burden_lr and embedding_lr each completed 5×10 outer folds. burden measures the number of prior diagnoses and hospitalizations; embedding adds the TF-IDF/SVD representation of prior ICD codes on top of burden, all fitted on the training subset.

| Model | AUROC fold mean | AUPRC pooled OOF mean | Brier pooled OOF mean | Calibration slope pooled OOF mean |
|---|---:|---:|---:|---:|
| static_lr | 0.7993 | 0.5948 | 0.1509 | 0.898 |
| burden_lr | 0.7994 | 0.5947 | 0.1509 | 0.892 |
| embedding_lr | 0.7959 | 0.5928 | 0.1523 | 0.830 |

- `burden_lr` vs `static_lr`: ΔAUROC=0.0001, Nadeau–Bengio corrected 95% CI [-0.0022, 0.0024], p=0.944, Holm-corrected p=1.000.
- `embedding_lr` vs `burden_lr`: ΔAUROC=-0.0035, Nadeau–Bengio corrected 95% CI [-0.0108, 0.0038], p=0.347, Holm-corrected p=1.000.

**Diagnosis-count features barely change performance, and the current embedding shows no gain.** The pooled OOF AUROC of embedding is lower than the burden comparator in all five repeats, and the point estimates of Brier and calibration slope are also worse; but the corrected interval of the difference crosses 0, so it cannot be written as a significant loss of predictive ability.

Implementation check: all 50 folds produced a 32-dimensional embedding, with no case of an empty vocabulary causing the whole representation to be skipped. This does not show that all comorbidity or text representations are useless; this experiment concerns only prior-hospitalization ICD codes, the current TF-IDF/SVD dimensions and fixed-parameter LR.

For now the static model and stacking are kept as the main candidates, and this ablation is reported as a complete negative result. Combined stacking is not finished yet; it can be the last pre-planned exploration of the original plan, but it should not be expected in advance to improve performance.
